# 🌾 Model 1: PlantVillage — Tarla Bitkileri (DÜZELTİLMİŞ)
## EfficientNetB3 · 38 Sınıf

**⚠️ Önceki versiyondaki `rescale=1./255` hatası düzeltildi.**
EfficientNetB3 kendi normalizasyonunu yapar, ekstra rescale gereksiz.

In [ ]:
# ── 1: GPU + Kurulum ─────────────────────────────────────
import tensorflow as tf
print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
!pip install tf2onnx onnx onnxruntime kagglehub scikit-learn seaborn -q
print('✅ Hazır')

In [ ]:
# ── 2: Drive + Veri Seti ─────────────────────────────────
import os, time, shutil, kagglehub
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/PlantDisease'
os.makedirs(DRIVE_BASE, exist_ok=True)

print('📦 PlantVillage indiriliyor...')
pv_path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
print(f'✅ İndirildi: {pv_path}')

TRAIN_DIR = VALID_DIR = None
for root, dirs, files in os.walk(pv_path):
    if 'train' in dirs:
        TRAIN_DIR = os.path.join(root, 'train')
        VALID_DIR = os.path.join(root, 'valid')
        break
print(f'Train: {TRAIN_DIR} ({len(os.listdir(TRAIN_DIR))} sınıf)')

In [ ]:
# ── 3: Yerel Diske Kopyala (I/O hızı için) ──────────────
LOCAL_TRAIN = '/content/data/train'
LOCAL_VALID = '/content/data/valid'

if not os.path.exists(LOCAL_TRAIN):
    print('📋 Yerel diske kopyalanıyor...')
    start = time.time()
    shutil.copytree(TRAIN_DIR, LOCAL_TRAIN)
    shutil.copytree(VALID_DIR, LOCAL_VALID)
    print(f'✅ Kopyalandı ({(time.time()-start)/60:.1f} dk)')
else:
    print('✅ Zaten kopyalanmış')

TRAIN_DIR = LOCAL_TRAIN
VALID_DIR = LOCAL_VALID

In [ ]:
# ── 4: Veri Üreteçleri (rescale YOK!) ────────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE, BATCH_SIZE = 224, 64

# ⚠️ rescale=1./255 KALDIRILDI!
# EfficientNet kendi preprocessing'ini yapar
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15, height_shift_range=0.15,
    shear_range=0.15, zoom_range=0.2,
    horizontal_flip=True, brightness_range=[0.8, 1.2],
    fill_mode='nearest'
).flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', seed=42
)

valid_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    VALID_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
class_names = [k for k,_ in sorted(train_gen.class_indices.items(), key=lambda x: x[1])]
with open('/content/plant_class_names.txt', 'w') as f:
    f.write('\n'.join(class_names))

print(f'\n✅ {NUM_CLASSES} sınıf | Eğitim: {train_gen.samples:,} | Doğrulama: {valid_gen.samples:,}')

In [ ]:
# ── 5: EfficientNetB3 Model ──────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3

base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224,224,3))
base.trainable = False

inp = keras.Input(shape=(224,224,3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = keras.Model(inp, out)
print(f'✅ {model.count_params():,} parametre')

In [ ]:
# ── 6: FAZ 1 — Dondurulmuş Eğitim ───────────────────────
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
H5 = os.path.join(DRIVE_BASE, 'plant_model.h5')

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

print('🚀 FAZ 1...')
h1 = model.fit(train_gen, epochs=15, validation_data=valid_gen, callbacks=[
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6),
    ModelCheckpoint(H5, monitor='val_accuracy', save_best_only=True, verbose=1),
])
print(f'\n✅ FAZ 1 → En iyi: {max(h1.history["val_accuracy"])*100:.1f}%')

In [ ]:
# ── 7: FAZ 2 — Fine-tuning ──────────────────────────────
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy', metrics=['accuracy'])

print('🔧 FAZ 2...')
h2 = model.fit(train_gen, epochs=30,
    initial_epoch=len(h1.history['accuracy']),
    validation_data=valid_gen, callbacks=[
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7),
    ModelCheckpoint(H5, monitor='val_accuracy', save_best_only=True, verbose=1),
])
print(f'\n✅ FAZ 2 → En iyi: {max(h2.history["val_accuracy"])*100:.1f}%')

In [ ]:
# ── 8: 📈 Eğitim Grafikleri ──────────────────────────────
import matplotlib.pyplot as plt

acc = h1.history['accuracy'] + h2.history['accuracy']
val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
loss = h1.history['loss'] + h2.history['loss']
val_loss = h1.history['val_loss'] + h2.history['val_loss']
epochs_r = range(1, len(acc)+1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#334155')
    ax.grid(True, alpha=0.15, color='white')

axes[0].plot(epochs_r, acc, color='#00d4aa', lw=2.5, label='Eğitim', marker='o', ms=4)
axes[0].plot(epochs_r, val_acc, color='#ff6b6b', lw=2.5, label='Doğrulama', marker='s', ms=4)
axes[0].axvline(x=len(h1.history['accuracy'])+0.5, color='#ffd700', ls='--', alpha=0.7, label='FAZ 2')
axes[0].set_title('📊 Doğruluk', color='white', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch', color='white')
axes[0].legend(facecolor='#0f3460', labelcolor='white')

axes[1].plot(epochs_r, loss, color='#00d4aa', lw=2.5, label='Eğitim', marker='o', ms=4)
axes[1].plot(epochs_r, val_loss, color='#ff6b6b', lw=2.5, label='Doğrulama', marker='s', ms=4)
axes[1].axvline(x=len(h1.history['accuracy'])+0.5, color='#ffd700', ls='--', alpha=0.7, label='FAZ 2')
axes[1].set_title('📉 Kayıp', color='white', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch', color='white')
axes[1].legend(facecolor='#0f3460', labelcolor='white')

best = max(val_acc); be = val_acc.index(best)+1
plt.suptitle(f'🌾 PlantVillage — En İyi: {best*100:.1f}% (Epoch {be})', color='white', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, 'plant_training_history.png'), dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
plt.show()

In [ ]:
# ── 9: 🎯 Hata Matrisi + Rapor ──────────────────────────
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

valid_gen.reset()
y_pred = np.argmax(model.predict(valid_gen, verbose=1), axis=1)
y_true = valid_gen.classes

print('\n📋 SINIFLANDIRMA RAPORU:')
print('='*70)
report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)
with open(os.path.join(DRIVE_BASE, 'plant_report.txt'), 'w') as f:
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
short_names = [n.replace('___',' ').replace('_',' ')[:20] for n in class_names]

fig, ax = plt.subplots(figsize=(24, 20))
fig.patch.set_facecolor('#1a1a2e')
sns.heatmap(cm, annot=False, cmap='YlOrRd', xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_title('🗺 Hata Matrisi', color='white', fontsize=16, fontweight='bold')
ax.tick_params(colors='white', labelsize=6)
plt.xticks(rotation=90); plt.yticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, 'plant_confusion_matrix.png'), dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
plt.show()

In [ ]:
# ── 10: 📊 Sınıf Bazlı Doğruluk ─────────────────────────
per_class = cm.diagonal() / cm.sum(axis=1)
colors = ['#22c55e' if a>0.9 else '#f59e0b' if a>0.7 else '#ef4444' for a in per_class]

fig, ax = plt.subplots(figsize=(16, 10))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')
ax.barh(range(len(class_names)), per_class*100, color=colors, height=0.7)
ax.set_yticks(range(len(class_names)))
ax.set_yticklabels(short_names, fontsize=7, color='white')
ax.set_xlabel('Doğruluk (%)', color='white')
ax.set_title('📊 Sınıf Bazlı Doğruluk', color='white', fontsize=14, fontweight='bold')
ax.axvline(x=90, color='#22c55e', ls='--', alpha=0.5)
ax.tick_params(colors='white'); ax.invert_yaxis(); ax.set_xlim(0,105)
for i,v in enumerate(per_class): ax.text(v*100+1, i, f'{v*100:.0f}%', va='center', color='white', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_BASE, 'plant_per_class.png'), dpi=150, facecolor='#1a1a2e', bbox_inches='tight')
plt.show()

In [ ]:
# ── 11: ONNX + İndirme ──────────────────────────────────
import tf2onnx, onnx, onnxruntime as ort

vl, va = model.evaluate(valid_gen, verbose=0)
ONNX_PATH = os.path.join(DRIVE_BASE, 'plant_model.onnx')

print('🔄 ONNX çeviriliyor...')
sig = [tf.TensorSpec(model.input_shape, tf.float32, name='input')]
om, _ = tf2onnx.convert.from_keras(model, input_signature=sig)
onnx.save(om, ONNX_PATH)

s = ort.InferenceSession(ONNX_PATH)
r = s.run(None, {s.get_inputs()[0].name: np.random.rand(1,224,224,3).astype(np.float32)})[0]
print(f'✅ ONNX OK → {r.shape}')

shutil.copy('/content/plant_class_names.txt', os.path.join(DRIVE_BASE, 'plant_class_names.txt'))
sz = os.path.getsize(ONNX_PATH)/(1024*1024)

print(f'\n{"="*50}')
print(f'✅ MODEL 1 TAMAMLANDI!')
print(f'{"="*50}')
print(f'🎯 {va*100:.1f}% | 📦 {sz:.0f} MB | 📊 {NUM_CLASSES} sınıf')

from google.colab import files
files.download(ONNX_PATH)
files.download('/content/plant_class_names.txt')